In [ ]:
# !pip install youtube-transcript-api

In [55]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.formatters import TextFormatter
from IPython.display import Markdown, display, update_display

In [31]:
#  Setting up variables

load_dotenv()

google_api_key = os.getenv('GOOGLE_API_KEY')
google_url = 'https://generativelanguage.googleapis.com/v1beta/openai/'

if not google_api_key:
    print('No api key found')



In [39]:
gemini = OpenAI(base_url=google_url, api_key=google_api_key)

ytt = YouTubeTranscriptApi()
formatter = TextFormatter()

In [76]:
system_prompt = """You are warm and helpful assistant of a content creator. You take the transcript and title of the creator's video and analyse it to create articles that motivate readers to watch the video. 
The content of the article must be warm, humourous, and inviting or serious, as per the tone of the transcript. The content should appeal to the relevant audience. 
All the content of the video should not be summarized. Identify the irrelevant or enlongated parts on same topics and add only a hint of it to keep the reader's intrigue.
Separate topics with relevant headings and topics. Recognize facts, theories or reasearch, and if a citation is not given, do not mention it in the article.
Any personal anecdotes or experiences mentioned should be checked as per relevance on topic and added. Group similar topics under same heding, especially if it is a continuous session. Ignore off topic statements.
The article should reflect the main theme and points of utmost interest without giving away all the content of the video.
Organize and respond in Markdown."""

In [34]:
def get_video_id(url):
    prefix, sep, suffix = url.partition('?v=')
    prefix, sep, suffix = suffix.partition('&')
    return prefix

In [40]:
def fetch_transcript_text(video_id):
    fetched_transcript = ytt.fetch(video_id)
    transcript_text = formatter.format_transcript(fetched_transcript)
    return transcript_text

In [45]:
def get_user_prompt(title,transcript):
    user_prompt = f"Following is the transcript of a video titled - {title}"
    user_prompt += "Read, analyse and make an promotion article to be posted on creator blog."
    user_prompt += f"Transcript: \n {transcript}"
    return user_prompt

In [60]:
def create_article(title, url):
    video_id = get_video_id(url)
    if not video_id:
        return 'Not a valid Youtube URL'

    transcript = fetch_transcript_text(video_id)
    if not transcript:
        return 'Transcript could not be fetched'

    user_prompt = get_user_prompt(title,transcript);
    message = [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': user_prompt}]
    
    response = gemini.chat.completions.create(model='gemini-3.1-flash-lite-preview', messages=message)

    return Markdown(response.choices[0].message.content)
    

In [74]:
display(create_article('Olivia Rodrigo Makes Chicken Lumpia | Now Serving | Vogue','https://www.youtube.com/watch?v=3sf3CbHJxpM'))

# Olivia Rodrigo’s Kitchen Confidential: Lumpia, Palomas, and Songwriting Secrets

Ever wondered what goes on in the mind of a superstar when she’s off-stage, craving comfort food, and trading stadium microphones for kitchen whisks? 

In her latest appearance on *Vogue’s "Now Serving,"* Olivia Rodrigo invites us into her kitchen to whip up a batch of her grandma’s classic chicken lumpia paired with a perfectly zesty, "slightly elevated" Paloma. It’s a candid, humorous, and heartfelt look at the woman behind the chart-topping anthems.

### A Recipe for "Big Magic"
While the lumpia sizzles and the tequila flows, Olivia opens up about her creative process. Far from the polished exterior of a world tour, she describes songwriting as a "download from the universe"—a flow state she finds most accessible when she’s completely alone. 

Whether she’s battling the tears brought on by freshly chopped onions or confessing that her creative breakthroughs often happen during the most mundane tasks (like doing the dishes), Olivia reminds us that art isn't always glamorous—it’s just persistent. She reveals a fascinating philosophy on quantity over quality: she has written roughly 10,000 songs in her life, knowing that even the "bad" ones serve as building blocks for the hits we love.

### From Stage Fright to Salted Rims
The conversation covers everything from the grueling reality of tour sleep schedules (and the secret power of ice baths) to the hilarious, relatable mishaps of being 19 and trying to order a cocktail you can barely pronounce. 

Olivia also touches on:
* **Tour Rituals:** Why she has to stand outside the dressing room door to "shake out the jitters" before hitting the stage.
* **The "Litmus Test":** How playing a song live acts as the ultimate filter—if the crowd jumps, the song works.
* **The Joy Shift:** After two albums defined by heartbreak and angst, she shares her intentional journey toward exploring joy and "sad love songs" in her recent work.

### Why You Need to Watch
Whether you’re a fan of her music or just looking for a new comfort food recipe to try on a cozy night in, this video is a perfect reminder that even our favorite icons have a messy, funny, and deeply human side. 

Olivia proves that you don’t need to be a professional chef to make something delicious—you just need a little bit of love, a lot of garlic, and perhaps a well-earned drink to toast your efforts.

**Ready to see Olivia trade her guitar for a frying pan? Watch the full episode of *Now Serving* on Vogue now!** 

***

*Have you tried making lumpia at home? Or do you have your own "secret" pre-show ritual? Let us know in the comments!*

In [73]:
def stream_article(title, url):
    video_id = get_video_id(url)
    if not video_id:
        return 'Not a valid Youtube URL'

    transcript = fetch_transcript_text(video_id)
    if not transcript:
        return 'Transcript could not be fetched'

    user_prompt = get_user_prompt(title,transcript);
    message = [{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': user_prompt}]

    stream = gemini.chat.completions.create(model='gemini-3.1-flash-lite-preview', messages=message, stream=True)
    response = ''
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [77]:
stream_article("SEVENTEEN (세븐틴) 5th Album 'HAPPY BURSTDAY' LISTENING SESSION",'https://www.youtube.com/watch?v=lX20qNA3ev8')

# A Peek Behind the Curtain: Inside SEVENTEEN’s 'HAPPY BURSTDAY' Listening Session

The atmosphere was electric, a little nervous, and filled with the kind of genuine warmth that only comes after ten years of growing up together. The members of SEVENTEEN recently gathered for a special listening session for their 5th album, **'HAPPY BURSTDAY'**, and the result was nothing short of a musical homecoming.

Forget the polished, formal reviews; this was an intimate look at the members baring their artistic souls. With WOOZI steering the ship as the producer and the members acting as each other’s biggest fans (and occasional music critics), the session turned into a journey through the evolution of who they are as artists today.

### The Art of Becoming "Me"
The core theme of this album is simple yet profound: **"Me."** Over a decade of performing as a unit, the members of SEVENTEEN have mastered the art of group harmony, but this album marks a deliberate pivot toward the individual. 

From THE 8’s venture into "emotional EDM" with "Skyfall," a genre he’s fallen in love with through his DJ lessons, to JOSHUA’s surprisingly refreshing and bright "Fortunate Change," the session was essentially a series of personal revelations. 

### A Candid Look at the Solo Sessions
It’s not just a collection of songs; it’s a collection of identities. Some highlights included:

*   **WONWOO’s "99.9%":** A rare, fun look at the "big cutie" we rarely see, which, according to WOOZI, finally captures the real side of him that he’s always kept hidden.
*   **SEUNGKWAN’s "Raindrops":** A track that surprised even his own bandmates, pushing his vocals into a raw, live-feeling space that feels like the climax of an emotional film.
*   **HOSHI’s "Damage":** A high-energy, early 2000s-inspired anthem featuring Timbaland that proves HOSHI knows exactly what he wants—and he isn't afraid to go after it.
*   **MINGYU’s "Shake It Off":** A track that caught everyone off guard with its cool, chill aesthetic—a total reflection of his personal vibe.
*   **DK’s "Happy Virus":** A deeply personal reflection on his ten-year journey. As WOOZI pointed out, it’s not just "happy"; it’s a song that turns his signature brightness into something truly moving.
*   **WOOZI’s "Destiny":** A special gift of a song that breaks away from his usual production style. For the first time, WOOZI stepped back, took a track written by long-time collaborators, and delivered a main vocal performance that even he found deeply emotional.
*   **VERNON’s "Shining Star":** A dive into "alien rock," a space-age sound that feels uniquely VERNON.
*   **JUN’s "Gemini":** A song reflecting on the duality of the inner and outer self—a concept he embraced while filming his recent series.
*   **DINO’s "Trigger":** A raw, unfiltered exploration of self-doubt and growth, anchored by the catchy opening "Bang!" that set the room on fire.
*   **JEONGHAN’s "Coincidence":** A track that perfectly balances his mischievous side with his sentimental heart, exploring the beauty of parting ways.
*   **S.COUPS’s "Jungle":** A powerful, stage-ready anthem that encapsulates exactly who he is as a leader and a performer.

### Beyond the Music
What makes this album so significant isn’t just the high quality of the tracks—it’s the vulnerability required to share them. As WOOZI noted, after a decade in the industry, it's easy to fall into a rhythm of habit. But this album? It was a return to their roots, a way to move beyond the expectations of being an "idol" and simply being *themselves*.

You can feel the pride in the room—the kind that comes from fourteen people who have navigated a decade of highs and lows, finally letting their individual colors shine. 

**Want to hear the stories and the songs for yourself?** 
This listening session wasn't just a rehearsal; it was an open door into the hearts of SEVENTEEN. Grab your headphones, find a quiet space, and get ready to meet the members in a way you never have before.

***

**Are you ready to join the journey? Watch the full 'HAPPY BURSTDAY' listening session [insert link here] and experience the growth, the laughs, and the raw talent of SEVENTEEN.**